# Sphere Stratification Comparison: SylverLining vs FastDer3Valent

Attribution notice: the fast-der solve-and-lift strategy used here follows companion work by Chris Liu, Joshua Maglione, and James B. Wilson.

This notebook uses exact lattice sphere tensors so their derivation laws survive randomization. It compares stratification with two derivation methods.


In [ ]:
using Pkg
Pkg.activate("../")

In [ ]:
using Random
using ITensors
using Plots
using Dleto

In [ ]:
Random.seed!(42)
include("../bench/SphereHarness.jl")

# Exact densor sphere support: i + j + k = d - 1. A thick numerical shell
# includes off-sphere points and need not admit nontrivial derivations.
make_sphere_tensor_itensor(d) = sphere_octant(d; valence=3)


In [ ]:
sphere_dims = (8, 10)
raw_spheres_itensor = [make_sphere_tensor_itensor(d) for d in sphere_dims]
println("Built $(length(raw_spheres_itensor)) exact sphere tensors as ITensors")
[dims(S) for S in raw_spheres_itensor]


In [ ]:
function run_pipeline_itensor(Γ_itensor)
    # Γ_itensor is already an ITensor from make_sphere_tensor_itensor
    
    println("\nRandomizing ITensor...")
    rn = randomize_tensor(Γ_itensor)
    
    println("Removing degeneracy...")
    nd = nondeg(rn.Δ)
    
    println("\nSylverLining Solver")
    @time base = stratify(nd.Δ; method=:SylverLining, tol=1e-6)
    
    println("\nFastDer3Valent Solver")
    @time fast = stratify(nd.Δ; method=:FastDer3Valent, tol=1e-10)
    
    return (;
        original=Γ_itensor,
        randomized=rn.Δ,
        nondeg=nd.Δ,
        strat_base=base.Σ,
        strat_fast=fast.Σ,
        base_changes=base.Xs,
        base_result=base,
        fast_changes=fast.Xs,
        fast_result=fast
    )
end

println("Running pipelines on both sphere tensors...")
results_itensor = [run_pipeline_itensor(S) for S in raw_spheres_itensor]
println("Completed $(length(results_itensor)) pipelines")

In [ ]:
function verify_derivations_detailed(res; method_name="Method")
    """
    Verify that the stratification result contains valid derivations.
    Check the number and dimensions of the derivation basis.
    """
    if method_name == "SylverLining"
        Xs = res.base_changes
        Σ = res.strat_base
        result = res.base_result
    else
        Xs = res.fast_changes
        Σ = res.strat_fast
        result = res.fast_result
    end
    
    println("\n=== $method_name ===")
    println("Stratified tensor dimension: $(dims(Σ))")
    println("Number of basis vectors: $(length(Xs))")
    
    if length(Xs) > 0
        println("Basis vector shapes:")
        for (idx, X) in enumerate(Xs[1:min(3, length(Xs))])
            try
                println("  Vector $idx: $(dims(X))")
            catch
                println("  Vector $idx: unable to determine dims")
            end
        end
        if length(Xs) > 3
            println("  ... and $(length(Xs) - 3) more")
        end
    end
    
    return length(Xs)
end

println("\n\n" * "="^60)
println("DERIVATION VERIFICATION")
println("="^60)

for (idx, res) in enumerate(results_itensor)
    println("\n### SPHERE $idx ###")
    base_count = verify_derivations_detailed(res; method_name="SylverLining")
    fast_count = verify_derivations_detailed(res; method_name="FastDer3Valent")
    
    println("\n--- COMPARISON ---")
    println("SylverLining derivations: $base_count")
    println("FastDer3Valent derivations: $fast_count")
    println("Difference: $(fast_count - base_count) (FastDer has $(fast_count - base_count > 0 ? "MORE" : "FEWER"))")
end

In [ ]:
println("\nFastDer restricted-basis diagnostic")
println("="^60)
println("The exact lattice spheres have derivations by construction.")
println("The cells below compare the two methods and check each FastDer residual.")


In [ ]:
using LinearAlgebra

"""
Check FastDer basis vectors against the implemented derivation equation.
In Dleto this is the densor map residual, equivalent to the tensor equation
sum_i Xs[i] * Γ = 0 (with the library's index relabeling convention).
"""
function verify_fastder_derivation_equation(Γ; tol=1e-10, atol=1e-7)
    Γcore = Γ isa Dleto.TensorSpace.TensorElement ? Dleto.TensorSpace.unwrap(Γ) : Γ
    fr = collect(inds(Γ))
    Ω = Dleto.IndTransverseOps(fr, Dleto.UniversalOp())
    ch = Dleto.UniversalChisel(length(fr))
    method = Dleto.FastDer3ValentMethod()

    rΩ, expand_map, ders = Dleto.derTrOpsReduced(method, Ω, ch, Γ; tol=tol, nd=-1)
    _, densor_map = Dleto.sylvesterLM(Ω, ch, Γcore)

    println("  basis size: $(size(ders, 2))")
    for j in 1:size(ders, 2)
        v_full = expand_map(ders[:, j])
        resid = densor_map * v_full
        l2res = norm(resid)
        linfres = maximum(abs, resid)
        println("    derivation $j: ||res||_2=$l2res, ||res||_∞=$linfres, pass=$(linfres <= atol)")
    end
end

println("\n" * "="^74)
println("FASTDER BASIS TEST: verify sum_i Xs[i]*Γ = 0 via densor residual")
println("="^74)

for (k, res) in enumerate(results_itensor)
    println("Sphere $k")
    verify_fastder_derivation_equation(res.nondeg; tol=1e-10, atol=1e-7)
end

In [ ]:
println("\nInspecting nondeg/randomize outputs for derivation-basis access...")
first_res = results_itensor[1]
println("typeof(first_res.randomized): ", typeof(first_res.randomized))
println("typeof(first_res.nondeg): ", typeof(first_res.nondeg))
println("dims(first_res.nondeg): ", dims(first_res.nondeg))

In [ ]:
function plot_comparison_itensor(res; title_prefix="Sphere")
    # Work directly with ITensors
    p1 = plot_tensor(res.original; title="$title_prefix: Original", color=:blue)
    p2 = plot_tensor(res.randomized; title="$title_prefix: Randomized", color=:red)
    p3 = plot_tensor(res.strat_base; title="$title_prefix: SylverLining", color=:green)
    p4 = plot_tensor(res.strat_fast; title="$title_prefix: FastDer3Valent", color=:orange)
    return plot(p1, p2, p3, p4; layout=(2, 2), size=(1000, 800))
end

In [ ]:
plot_comparison_itensor(results_itensor[1]; title_prefix="Sphere 1")

In [ ]:
plot_comparison_itensor(results_itensor[2]; title_prefix="Sphere 2")